# Scorecard — return **and** risk for every strategy, on one basis

The individual reports used slightly different assumptions: v1 let cash earn 0% and measured Sharpe over 0%, while v2 and v3 used T-bills. This notebook puts every strategy on **one basis**, so they can be compared side by side:

- $100,000 start; 0.10% fee + 0.05% slippage per side; decisions after the close, filled at the next open;
- **cash earns the 13-week T-bill rate** (lagged one day), and **Sharpe is measured over T-bills**;
- metrics: final value, annual return (CAGR), annual volatility, **maximum drawdown**, Sharpe, Calmar (CAGR ÷ |max drawdown|), round trips, trading costs, time invested, and the share of 1,000 random-timing traders beaten (same time invested, same number of switches);
- every version is shown deciding **daily** (a fresh 5-day prediction each trading day, the original design) and **weekly** (after Friday's close for the next 5 days, the brief's literal reading).

Frozen models only; nothing is selected here. With T-bill cash, v1's final value differs slightly from its own report, which assumed 0% cash.

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd

from src import config, v2, v3, weekly
from src.backtest import to_positions
from src.data import load_snapshot
from src.dataset import build_dataset_v2

raw = load_snapshot(); px = raw[["Open", "Close"]]
rf = v2.load_tbill_daily(raw.index)
HOLD = config.ROOT / "data" / "holdout"
raw_h = pd.read_csv(HOLD / "spy_20261013.csv", index_col=0, parse_dates=True); px_h = raw_h[["Open", "Close"]]
rf_h = v2.load_tbill_daily(raw_h.index, HOLD / "irx_20261013.csv")

ROWS = []
def score(period, strategy, schedule, held, prices, rfs, random_n=1000):
    rfs = rfs.loc[held.index]
    sim = v2.simulate(held, prices.loc[held.index], rfs)
    s = v2.summarize(sim, held, rfs, "")
    pct = None
    if s["round_trips"] > 1 and random_n:
        rand = v2.random_timing_benchmark(held, prices.loc[held.index], rfs, n=random_n)
        pct = round(float((rand < sim["value"].iloc[-1]).mean()), 3)
    ROWS.append({"period": period, "strategy": strategy, "schedule": schedule,
                 "final_value": s["final_value"], "cagr": s["cagr"], "ann_vol": s["ann_vol"],
                 "max_drawdown": s["max_drawdown"], "sharpe": s["sharpe_excess"],
                 "calmar": round(s["cagr"] / abs(s["max_drawdown"]), 2) if s["max_drawdown"] < 0 else None,
                 "round_trips": s["round_trips"], "costs": s["total_fees"], "time_invested": s["exposure"],
                 "beats_random": pct})

def both(period, strategy, daily_pos, weekly_pos, start, prices, rfs, random_n=1000):
    score(period, strategy, "daily", v2.held_from_positions(daily_pos, start=start), prices, rfs, random_n)
    score(period, strategy, "weekly", v2.held_from_positions(weekly_pos, start=start), prices, rfs, random_n)

def buy_and_hold(period, index, prices, rfs):
    score(period, "Buy & Hold", "—", v2.buy_and_hold_held(index), prices, rfs, 0)

## 2023–2024 test (seen by every version)

In [2]:
P = "2023–24 test"
p1 = pd.read_csv(config.OUTPUT_DIR / "test_predictions.csv", index_col=0, parse_dates=True)
buy_and_hold(P, p1.index, px, rf)
both(P, "v1 · random forest", to_positions(p1["action"], min_hold=5),
     weekly.weekly_positions_from_actions(p1["action"], start=0), 0, px, rf)
er2 = pd.read_csv(config.OUTPUT_DIR / "v2" / "test_predictions.csv", index_col=0, parse_dates=True)["E_r"]
both(P, "v2 · gradient boosting", v2.positions_from_er(er2, "long"),
     weekly.weekly_positions_from_er(er2, "long"), 1, px, rf)
ds = build_dataset_v2(raw)
er3 = v3.expected_return(v3.walk_forward_proba(ds, "C3", [2023, 2024], v3.v2_params()))
both(P, "v3 · calibrated odds", v2.positions_from_er(er3, "long"),
     weekly.weekly_positions_from_er(er3, "long"), 1, px, rf)

## 2013–2024 walk-forward (yearly refits; reporting only)

In [3]:
P = "2013–24 walk-forward"
years = list(range(2013, 2025))
er_wf2 = v3.expected_return(v3.walk_forward_proba(ds, "C0", years, v3.v2_params()))
buy_and_hold(P, er_wf2.index, px, rf)
both(P, "v2 · gradient boosting", v2.positions_from_er(er_wf2, "long"),
     weekly.weekly_positions_from_er(er_wf2, "long"), 1, px, rf, random_n=200)
er_wf3 = v3.expected_return(v3.walk_forward_proba(ds, "C3", years, v3.v2_params()))
both(P, "v3 · calibrated odds", v2.positions_from_er(er_wf3, "long"),
     weekly.weekly_positions_from_er(er_wf3, "long"), 1, px, rf, random_n=200)

## 2025–26 holdout (v3's primary holdout; v1 and v2 were not run here)

In [4]:
P = "2025–26 holdout"
er_h = v3.expected_return(v3.walk_forward_proba(build_dataset_v2(raw_h), "C3", [2025, 2026], v3.v2_params()))
buy_and_hold(P, er_h.index, px_h, rf_h)
both(P, "v3 · calibrated odds", v2.positions_from_er(er_h, "long"),
     weekly.weekly_positions_from_er(er_h, "long"), 1, px_h, rf_h)

## The scorecard

In [5]:
card = pd.DataFrame(ROWS)
fmt = {"final_value": "${:,.0f}", "cagr": "{:.1%}", "ann_vol": "{:.1%}", "max_drawdown": "{:.1%}",
       "sharpe": "{:.2f}", "calmar": "{:.2f}", "costs": "${:,.0f}", "time_invested": "{:.0%}", "beats_random": "{:.0%}"}
for period, g in card.groupby("period", sort=False):
    print(period)
    display(g.drop(columns="period").set_index(["strategy", "schedule"]).style.format(fmt, na_rep="—"))
(config.OUTPUT_DIR / "scorecard.json").write_text(json.dumps(json.loads(card.to_json(orient="records")), indent=2))
print("saved outputs/scorecard.json")

2023–24 test


2013–24 walk-forward


2025–26 holdout


saved outputs/scorecard.json
